In [1]:
import xarray as xr
from pathlib import Path
import matplotlib.pyplot as plt
from cartopy import crs as ccrs # Cartography library
import numpy as np
import seaborn as sns
import geopy.distance
import re
import pandas as pd
import cartopy.feature as cfeature
from datetime import timedelta
import yaml

In [2]:
def to_dt(dt64):
    return pd.Timestamp(dt64).to_pydatetime(warn=False)

In [3]:
def ddm2dec(dms_str):
    """Return decimal representation of DDM (degree decimal minutes)
    
    >>> ddm2dec("45° 17,896' N")
    48.8866111111F
    """
    
    dms_str = re.sub(r'\s', '', dms_str)
    
    sign = -1 if re.search('[swSW]', dms_str) else 1
    
    numbers = [*filter(len, re.split('\D+', dms_str, maxsplit=4))]

    degree = numbers[0]
    minute_decimal = numbers[1] 
    decimal_val = numbers[2] if len(numbers) > 2 else '0' 
    minute_decimal += "." + decimal_val

    return sign * (int(degree) + float(minute_decimal) / 60)

<>:12: SyntaxWarning: invalid escape sequence '\D'
<>:12: SyntaxWarning: invalid escape sequence '\D'
/var/folders/l3/7mstk0nx3_3c69r4z8lzlp0w0000gp/T/ipykernel_5468/1444069316.py:12: SyntaxWarning: invalid escape sequence '\D'
  numbers = [*filter(len, re.split('\D+', dms_str, maxsplit=4))]


In [6]:
ipfs_root_log= "ipfs://bafybeianebwhw4uzkqnaekl5kyoau7hubxaens7azrftgqtz2mccciejle"

In [7]:
file_log = f"{ipfs_root_log}/met_203_1_station_book.csv"
### Read data log
data_raw = pd.read_csv(file_log,header=0,sep=';',usecols=[6,7,8, 9,10, 11, 12, 13, 17],dtype={'Event Time': str, 'Event Comment': str,'Action': str})
data_raw['time']= pd.to_datetime(data_raw.pop('Event Time'),format ='%d.%m.%y %H:%M')
data_raw = data_raw.set_index('time').sort_index()

data_raw = data_raw.rename(columns={'Latitude':'lat','Longitude':'lon'})
data_raw['lat'] = [ddm2dec(data_raw['lat'].iloc[i]) for i in range(data_raw.shape[0])]
data_raw['lon'] = [ddm2dec(data_raw['lon'].iloc[i]) for i in range(data_raw.shape[0])]

In [8]:
ctd = data_raw[(data_raw['Device Shortname']=='CTD') & (
    (data_raw['Action'].str.contains("water")) |(data_raw['Action'].str.contains("inform")) | (data_raw['Action'].str.contains('deck')))]

In [9]:
ctd_ini = ctd[(ctd['Action'].str.contains("water")) | (ctd['Action'].str.contains('information') & ctd['Event Comment'].str.contains('W3'))]
ctd_end = ctd[(ctd['Action'].str.contains("deck"))]

In [10]:
ctd_resume = ctd_ini[['Device Shortname','lat','lon']]

In [11]:
ctd_resume['time_end'] = ctd_end.index.values
ctd_resume['duration(min)'] = np.array(((ctd_end.index - ctd_ini.index).total_seconds()/60))

/var/folders/l3/7mstk0nx3_3c69r4z8lzlp0w0000gp/T/ipykernel_5468/2135117569.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ctd_resume['time_end'] = ctd_end.index.values
/var/folders/l3/7mstk0nx3_3c69r4z8lzlp0w0000gp/T/ipykernel_5468/2135117569.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ctd_resume['duration(min)'] = np.array(((ctd_end.index - ctd_ini.index).total_seconds()/60))


In [12]:
mss = data_raw[(data_raw['Device Shortname']=='MSS')]

mss_ini = mss[mss['Action'].str.contains('water') | mss['Action'].str.contains('start')]
mss_end = mss[~(mss['Action'].str.contains('water') | mss['Action'].str.contains('start'))]
mss_resume = mss_ini[['Device Shortname','lat','lon']]
mss_resume['time_end'] = mss_end.index.values
mss_resume['duration(min)'] = np.array(((mss_end.index - mss_ini.index).total_seconds()/60))
mss_no_calibration = mss_resume[~(mss_ini['Event Comment'].isin(['re-calibration']).values | mss_end['Event Comment'].isin(['re-calibration']).values)]

/var/folders/l3/7mstk0nx3_3c69r4z8lzlp0w0000gp/T/ipykernel_5468/1245354922.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mss_resume['time_end'] = mss_end.index.values
/var/folders/l3/7mstk0nx3_3c69r4z8lzlp0w0000gp/T/ipykernel_5468/1245354922.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mss_resume['duration(min)'] = np.array(((mss_end.index - mss_ini.index).total_seconds()/60))


In [13]:
uav= data_raw[data_raw['Device Shortname']=='UAV']
uav_ini = uav[uav['Action'].str.contains('start')]
uav_end = uav[uav['Action'].str.contains('end')]
uav_resume = uav_ini[['Device Shortname','lat','lon']].iloc[:-1]
uav_resume['time_end'] = uav_end.index.values
uav_resume['duration(min)'] = np.array(((uav_end.index - uav_resume.index).total_seconds()/60))

In [14]:
def to_yaml(platform,df,moving='Stationary'):
    return {"mission": "ORCESTRA",
           "platform": platform,
           "stations": [{"kinds" : [moving],
                        "station_id" : name+'-'+ str(idx+1),
                        "type" : name,
                        "start" : to_dt(df.index[idx]),
                        "end" : to_dt(df.time_end.iloc[idx]),
                       } for idx,name in enumerate(df['Device Shortname'])
    ],}         

In [15]:
yaml.dump(to_yaml('METEOR',ctd_resume),
          open(f"/Users/hans/Documents/bow_tie/actionlog_book/stations_bowtie_ctd.yaml", "w"),
          sort_keys=False)

In [16]:
yaml.dump(to_yaml('METEOR',uav_resume),
          open(f"/Users/hans/Documents/bow_tie/actionlog_book/stations_bowtie_uav.yaml", "w"),
          sort_keys=False)

In [17]:
yaml.dump(to_yaml('METEOR',mss_resume,moving='Mobile'),
          open(f"/Users/hans/Documents/bow_tie/actionlog_book/stations_bowtie_mss.yaml", "w"),
          sort_keys=False)

In [4]:
with open("/Users/hans/Documents/bow_tie/actionlog_book/stations_bowtie_ctd.yaml", 'r') as stream:
    data_loaded = yaml.safe_load(stream)